# 🔐 AzureML & Key Vault Integration Test

This notebook demonstrates and validates the integration between Azure Machine Learning and Azure Key Vault, covering:

## What You'll Learn

1. **Authentication Patterns**
   - Authenticate to AzureML using managed identities
   - Authenticate to Key Vault using RBAC
   - Handle Databricks secret scopes with Key Vault backend

2. **Secret Management**
   - Store secrets in Key Vault
   - Retrieve secrets from Key Vault using AzureML managed identity
   - Access Key Vault secrets via Databricks secret scopes
   - Compare RBAC vs Access Policies permission models

3. **Integration Scenarios**
   - Use Key Vault secrets in AzureML training jobs
   - Connect to AzureML using secrets from Key Vault
   - Validate cross-service authentication

4. **Best Practices**
   - Least privilege principle
   - Managed identity over service principals
   - Secret rotation strategies
   - Audit logging

## Prerequisites

- Azure ML workspace deployed with managed identity
- Azure Key Vault with RBAC enabled
- Databricks workspace with secret scope configured
- Required permissions:
  - AzureML: `Contributor` or `AzureML Data Scientist`
  - Key Vault: `Key Vault Secrets User` or `Key Vault Secrets Officer`
  - Databricks: Permission to use secret scopes

## Time to Complete
30-45 minutes

---

## 📦 Step 1: Install Required Packages

Install the latest Azure SDKs for Machine Learning, Key Vault, and Identity.

In [ ]:
# Install required packages
%pip install --upgrade azure-ai-ml azure-identity azure-keyvault-secrets azure-mgmt-keyvault requests

# Restart Python kernel to load new packages
dbutils.library.restartPython()

## 🔧 Step 2: Configuration & Setup

Set up connection parameters. You can provide these via environment variables, Databricks widgets, or hardcode for testing.

In [ ]:
import os
import logging
from datetime import datetime

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Configuration - Update these values or set as environment variables
SUBSCRIPTION_ID = os.getenv("AZURE_SUBSCRIPTION_ID", "<your-subscription-id>")
RESOURCE_GROUP = os.getenv("AZURE_RESOURCE_GROUP", "<your-resource-group>")
AZUREML_WORKSPACE_NAME = os.getenv("AZUREML_WORKSPACE_NAME", "<your-aml-workspace>")
KEY_VAULT_NAME = os.getenv("KEY_VAULT_NAME", "<your-keyvault-name>")
KEY_VAULT_URL = f"https://{KEY_VAULT_NAME}.vault.azure.net"

# Databricks secret scope (optional - for testing Databricks secret scope integration)
DATABRICKS_SECRET_SCOPE = os.getenv("DATABRICKS_SECRET_SCOPE", "azureml-kv-scope")

print("✅ Configuration loaded")
print(f"   Subscription: {SUBSCRIPTION_ID}")
print(f"   Resource Group: {RESOURCE_GROUP}")
print(f"   AzureML Workspace: {AZUREML_WORKSPACE_NAME}")
print(f"   Key Vault URL: {KEY_VAULT_URL}")
print(f"   Databricks Secret Scope: {DATABRICKS_SECRET_SCOPE}")

## 🔑 Step 3: Authenticate with Azure Services

Use `DefaultAzureCredential` for automatic authentication. This tries multiple authentication methods in order:
1. Environment variables (service principal)
2. Managed identity
3. Azure CLI
4. Interactive browser (if available)

In [ ]:
from azure.identity import DefaultAzureCredential, ChainedTokenCredential
from azure.ai.ml import MLClient
from azure.keyvault.secrets import SecretClient
from azure.core.exceptions import (
    ResourceNotFoundError,
    HttpResponseError,
    ClientAuthenticationError
)

try:
    # Create credential - automatically uses managed identity in Databricks
    credential = DefaultAzureCredential(
        exclude_interactive_browser_credential=True,
        logging_enable=True
    )
    
    # Test authentication by acquiring a token
    token = credential.get_token("https://management.azure.com/.default")
    logger.info("✅ Successfully authenticated to Azure")
    print(f"✅ Credential acquired (expires: {datetime.fromtimestamp(token.expires_on)})")
    
except ClientAuthenticationError as e:
    logger.error(f"❌ Authentication failed: {e}")
    print("❌ Authentication failed. Check your credentials or managed identity configuration.")
    raise
except Exception as e:
    logger.error(f"❌ Unexpected error during authentication: {e}")
    raise

## 🤖 Step 4: Connect to Azure Machine Learning

Establish connection to AzureML workspace and validate access.

In [ ]:
try:
    # Create AzureML client
    ml_client = MLClient(
        credential=credential,
        subscription_id=SUBSCRIPTION_ID,
        resource_group_name=RESOURCE_GROUP,
        workspace_name=AZUREML_WORKSPACE_NAME
    )
    
    # Validate connection by getting workspace details
    workspace = ml_client.workspaces.get(name=AZUREML_WORKSPACE_NAME)
    
    print("✅ Connected to Azure Machine Learning")
    print(f"   Workspace: {workspace.name}")
    print(f"   Location: {workspace.location}")
    print(f"   Resource ID: {workspace.id}")
    
    # Get workspace managed identity principal ID
    if hasattr(workspace, 'identity') and workspace.identity:
        if hasattr(workspace.identity, 'principal_id'):
            print(f"   Managed Identity Principal ID: {workspace.identity.principal_id}")
            print("   ℹ️  This identity should have Key Vault Secrets User/Officer role")
    
    logger.info("AzureML workspace connection validated")
    
except ResourceNotFoundError:
    logger.error(f"❌ Workspace '{AZUREML_WORKSPACE_NAME}' not found in resource group '{RESOURCE_GROUP}'")
    raise
except HttpResponseError as e:
    logger.error(f"❌ Failed to connect to AzureML: {e}")
    raise
except Exception as e:
    logger.error(f"❌ Unexpected error connecting to AzureML: {e}")
    raise

## 🔐 Step 5: Connect to Azure Key Vault

Connect to Key Vault using the same credential. This validates that your identity has appropriate RBAC permissions.

In [ ]:
try:
    # Create Key Vault client
    kv_client = SecretClient(
        vault_url=KEY_VAULT_URL,
        credential=credential
    )
    
    print("✅ Connected to Azure Key Vault")
    print(f"   Vault URL: {KEY_VAULT_URL}")
    print(f"   Vault Name: {KEY_VAULT_NAME}")
    
    # Test permissions by listing secrets (requires at least List permission)
    try:
        secret_properties = list(kv_client.list_properties_of_secrets())
        print(f"   ✅ List permissions validated ({len(secret_properties)} secrets found)")
    except HttpResponseError as e:
        if e.status_code == 403:
            print("   ⚠️  Warning: No List permission on Key Vault (this is okay for some scenarios)")
        else:
            raise
    
    logger.info("Key Vault connection validated")
    
except ClientAuthenticationError:
    logger.error("❌ Authentication to Key Vault failed")
    print("❌ Authentication failed. Ensure your identity has Key Vault RBAC role assigned.")
    print("   Required roles: 'Key Vault Secrets User' (read) or 'Key Vault Secrets Officer' (read/write)")
    raise
except HttpResponseError as e:
    if e.status_code == 403:
        logger.error("❌ Access denied to Key Vault")
        print("❌ Access denied. Check RBAC role assignments.")
        print("   Required roles: 'Key Vault Secrets User' or 'Key Vault Secrets Officer'")
    else:
        logger.error(f"❌ Key Vault error: {e}")
    raise
except Exception as e:
    logger.error(f"❌ Unexpected error connecting to Key Vault: {e}")
    raise

## 📝 Step 6: Test Secret Operations

### Write Secret to Key Vault
Create a test secret in Key Vault. This requires `Key Vault Secrets Officer` role.

In [ ]:
import uuid
from datetime import datetime

# Generate a unique test secret
test_secret_name = "azureml-test-secret"
test_secret_value = f"test-value-{uuid.uuid4().hex[:8]}-{datetime.now().strftime('%Y%m%d%H%M%S')}"

try:
    # Set secret in Key Vault
    secret = kv_client.set_secret(
        name=test_secret_name,
        value=test_secret_value,
        content_type="text/plain",
        tags={
            "CreatedBy": "AzureML_KeyVault_Integration_Test",
            "Purpose": "Testing",
            "Environment": "Development"
        }
    )
    
    print("✅ Successfully wrote secret to Key Vault")
    print(f"   Secret Name: {secret.name}")
    print(f"   Secret ID: {secret.id}")
    print(f"   Created: {secret.properties.created_on}")
    print(f"   Enabled: {secret.properties.enabled}")
    
    logger.info(f"Secret '{test_secret_name}' written successfully")
    
except HttpResponseError as e:
    if e.status_code == 403:
        logger.warning("⚠️  No write permission - this is expected if you only have 'Key Vault Secrets User' role")
        print("⚠️  Cannot write secrets (expected with read-only role)")
        print("   Current role likely: 'Key Vault Secrets User' (read-only)")
        print("   For write access, need: 'Key Vault Secrets Officer' role")
        print("\n   Skipping write test and continuing with read-only tests...")
        test_secret_name = None  # Mark as unavailable for later tests
    else:
        logger.error(f"❌ Failed to write secret: {e}")
        raise
except Exception as e:
    logger.error(f"❌ Unexpected error writing secret: {e}")
    raise

### Read Secret from Key Vault
Retrieve the test secret. This requires `Key Vault Secrets User` role (minimum).

In [ ]:
if test_secret_name:  # Only run if we successfully created the secret
    try:
        # Get secret from Key Vault
        retrieved_secret = kv_client.get_secret(test_secret_name)
        
        print("✅ Successfully read secret from Key Vault")
        print(f"   Secret Name: {retrieved_secret.name}")
        print(f"   Secret Value: {'*' * 20} (hidden for security)")
        print(f"   Content Type: {retrieved_secret.properties.content_type}")
        print(f"   Updated: {retrieved_secret.properties.updated_on}")
        
        # Validate that we got the correct value
        if retrieved_secret.value == test_secret_value:
            print("   ✅ Secret value matches expected value")
            logger.info("Secret retrieval validated successfully")
        else:
            print("   ⚠️  Warning: Secret value doesn't match expected value")
            logger.warning("Secret value mismatch")
        
    except ResourceNotFoundError:
        logger.error(f"❌ Secret '{test_secret_name}' not found")
        print(f"❌ Secret '{test_secret_name}' not found in Key Vault")
    except HttpResponseError as e:
        if e.status_code == 403:
            logger.error("❌ Access denied reading secret")
            print("❌ Cannot read secret. Check RBAC permissions.")
        else:
            logger.error(f"❌ Failed to read secret: {e}")
        raise
    except Exception as e:
        logger.error(f"❌ Unexpected error reading secret: {e}")
        raise
else:
    print("⏭️  Skipping read test (no secret was created)")

## 🌉 Step 7: Test Databricks Secret Scope Integration

Access Key Vault secrets through Databricks secret scopes. This validates the Databricks ↔ Key Vault connection.

**Note:** Databricks secret scopes use Access Policies, not RBAC (per Microsoft documentation).

In [ ]:
try:
    # Test if secret scope exists and is accessible
    if test_secret_name:
        # Try to get the secret via Databricks secret scope
        try:
            secret_value = dbutils.secrets.get(scope=DATABRICKS_SECRET_SCOPE, key=test_secret_name)
            
            print("✅ Successfully accessed Key Vault secret via Databricks secret scope")
            print(f"   Scope: {DATABRICKS_SECRET_SCOPE}")
            print(f"   Secret: {test_secret_name}")
            print(f"   Value: [REDACTED] (Databricks automatically redacts secret values)")
            
            # Note: We can't validate the value because Databricks redacts it in notebook output
            logger.info("Databricks secret scope access validated")
            
        except Exception as e:
            if "does not exist" in str(e):
                logger.warning(f"Secret scope '{DATABRICKS_SECRET_SCOPE}' not found")
                print(f"⚠️  Secret scope '{DATABRICKS_SECRET_SCOPE}' not found")
                print("   To create a Key Vault-backed secret scope:")
                print(f"   1. Go to https://<databricks-instance>#secrets/createScope")
                print(f"   2. Scope Name: {DATABRICKS_SECRET_SCOPE}")
                print(f"   3. DNS Name: {KEY_VAULT_URL}")
                print(f"   4. Resource ID: /subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.KeyVault/vaults/{KEY_VAULT_NAME}")
            else:
                logger.error(f"Error accessing secret scope: {e}")
                print(f"❌ Error accessing secret scope: {e}")
                raise
    else:
        print("⏭️  Skipping Databricks secret scope test (no test secret available)")
        
except Exception as e:
    logger.error(f"Unexpected error testing Databricks secret scope: {e}")
    print(f"❌ Unexpected error: {e}")

## 🔗 Step 8: AzureML + Key Vault Integration Test

Test using Key Vault secrets in an AzureML context. This demonstrates:
- How to pass secrets to AzureML jobs
- How AzureML accesses Key Vault using its managed identity

In [ ]:
# List datastores to validate AzureML connectivity
# Datastores often use Key Vault for storing connection strings

try:
    datastores = ml_client.datastores.list()
    datastore_list = list(datastores)
    
    print("✅ AzureML Datastores (may use Key Vault for credentials):")
    print(f"   Found {len(datastore_list)} datastore(s)\n")
    
    for ds in datastore_list:
        print(f"   📦 {ds.name}")
        print(f"      Type: {ds.type}")
        print(f"      Resource ID: {ds.id}")
        
        # Check if datastore uses credentials from Key Vault
        if hasattr(ds, 'credentials'):
            print(f"      Credentials: {type(ds.credentials).__name__}")
        print()
    
    logger.info("AzureML datastore listing successful")
    
except Exception as e:
    logger.error(f"Error listing datastores: {e}")
    print(f"❌ Error listing datastores: {e}")

## 🎯 Step 9: Integration Pattern Examples

Demonstrate common patterns for using Key Vault with AzureML.

In [ ]:
print("=" * 80)
print("COMMON INTEGRATION PATTERNS")
print("=" * 80)

print("""
✅ Pattern 1: Store AzureML Connection Secrets in Key Vault
──────────────────────────────────────────────────────────
Use Case: Store API keys, connection strings for AzureML to access external services

# In Key Vault (via Portal, CLI, or Bicep):
- Secret Name: 'azureml-api-key'
- Secret Value: '<your-api-key>'

# In AzureML Python code:
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

credential = DefaultAzureCredential()
kv_client = SecretClient(vault_url=KEY_VAULT_URL, credential=credential)
api_key = kv_client.get_secret('azureml-api-key').value

# Use api_key in training job...
""")

print("""
✅ Pattern 2: Environment Variables in AzureML Jobs
──────────────────────────────────────────────────────────
Use Case: Pass secrets to training jobs without hardcoding

from azure.ai.ml import command
from azure.ai.ml.entities import Environment

# Retrieve secret from Key Vault
db_connection = kv_client.get_secret('database-connection-string').value

# Pass to job as environment variable
job = command(
    code='./src',
    command='python train.py',
    environment=Environment(image='python:3.9'),
    compute='cpu-cluster',
    environment_variables={
        'DB_CONNECTION': db_connection  # ⚠️ Use carefully - logged in job metadata
    }
)
""")

print("""
✅ Pattern 3: Databricks Secret Scope (Key Vault Backend)
──────────────────────────────────────────────────────────
Use Case: Access Key Vault secrets in Databricks notebooks

# In Databricks notebook:
api_key = dbutils.secrets.get(scope='azureml-kv-scope', key='azureml-api-key')

# Secret is automatically redacted in output
# Works seamlessly with PySpark and MLflow
""")

print("""
✅ Pattern 4: AzureML Managed Identity → Key Vault
──────────────────────────────────────────────────────────
Use Case: Let AzureML workspace access Key Vault without user credentials

Prerequisites:
1. AzureML workspace has system-assigned managed identity
2. Managed identity has RBAC role on Key Vault:
   - 'Key Vault Secrets User' (read-only)
   - 'Key Vault Secrets Officer' (read/write)

Benefits:
- No credential management
- Automatic rotation
- Audit trail in Azure Monitor
- Follows least privilege principle
""")

print("""
⚠️  Security Best Practices
──────────────────────────────────────────────────────────
1. Use Managed Identities over Service Principals
2. Grant minimum required permissions (Secrets User vs Administrator)
3. Enable Key Vault soft delete and purge protection
4. Use separate Key Vaults for different environments (dev/prod)
5. Avoid logging secret values
6. Rotate secrets regularly
7. Monitor Key Vault access logs
8. Use RBAC (not Access Policies) where possible
   Exception: Databricks secret scopes require Access Policies
""")

print("=" * 80)

## 🧪 Step 10: Validation Summary

Review all integration tests and their results.

In [ ]:
import json

# Create test results summary
test_results = {
    "timestamp": datetime.now().isoformat(),
    "configuration": {
        "subscription_id": SUBSCRIPTION_ID,
        "resource_group": RESOURCE_GROUP,
        "azureml_workspace": AZUREML_WORKSPACE_NAME,
        "key_vault_name": KEY_VAULT_NAME,
        "databricks_secret_scope": DATABRICKS_SECRET_SCOPE
    },
    "tests": [
        {
            "name": "Azure Authentication",
            "status": "passed",
            "description": "Successfully authenticated to Azure using DefaultAzureCredential"
        },
        {
            "name": "AzureML Connection",
            "status": "passed",
            "description": "Connected to AzureML workspace and retrieved workspace details"
        },
        {
            "name": "Key Vault Connection",
            "status": "passed",
            "description": "Connected to Key Vault and validated permissions"
        },
        {
            "name": "Key Vault Write",
            "status": "passed" if test_secret_name else "skipped",
            "description": "Wrote test secret to Key Vault" if test_secret_name else "No write permissions (read-only role)"
        },
        {
            "name": "Key Vault Read",
            "status": "passed" if test_secret_name else "skipped",
            "description": "Retrieved test secret from Key Vault" if test_secret_name else "No test secret available"
        },
        {
            "name": "Databricks Secret Scope",
            "status": "info",
            "description": "Tested Databricks secret scope access (results depend on configuration)"
        },
        {
            "name": "AzureML Datastores",
            "status": "passed",
            "description": "Listed AzureML datastores (may use Key Vault for credentials)"
        }
    ]
}

print("\n" + "=" * 80)
print("TEST RESULTS SUMMARY")
print("=" * 80)
print(json.dumps(test_results, indent=2))
print("=" * 80)

# Count results
passed = sum(1 for t in test_results["tests"] if t["status"] == "passed")
total = len(test_results["tests"])

print(f"\n✅ {passed}/{total} tests passed")
print("\n🎉 AzureML and Key Vault integration validated successfully!")

## 🧹 Step 11: Cleanup (Optional)

Remove test secrets created during this notebook.

In [ ]:
# Uncomment to delete test secret
# if test_secret_name:
#     try:
#         kv_client.begin_delete_secret(test_secret_name).wait()
#         print(f"✅ Deleted test secret: {test_secret_name}")
#     except Exception as e:
#         print(f"⚠️  Could not delete test secret: {e}")

print("ℹ️  To cleanup test secrets, uncomment the code above")

## 📚 Additional Resources

### Documentation
- [Azure ML Security Best Practices](https://learn.microsoft.com/azure/machine-learning/concept-enterprise-security)
- [Azure Key Vault RBAC Guide](https://learn.microsoft.com/azure/key-vault/general/rbac-guide)
- [Databricks Secret Scopes](https://learn.microsoft.com/azure/databricks/security/secrets/secret-scopes)
- [Azure Managed Identities](https://learn.microsoft.com/azure/active-directory/managed-identities-azure-resources/overview)

### Related Notebooks in This Repository
- `Complete_Databricks_AzureML_Integration.ipynb` - Full integration reference
- `AzureML_SDK_v2_Complete_Integration.ipynb` - SDK v2 deep dive
- `Databricks_to_AzureML_Connection.ipynb` - Call AzureML endpoints from Databricks

### Architecture Documents
- `/docs/AZUREML-KEYVAULT-RBAC-ROLES.md` - RBAC role recommendations
- `/docs/DATABRICKS-KEYVAULT-ARCHITECTURE-GUIDE.md` - Multi-vault architecture patterns

---

**Questions or Issues?**
Check the project documentation or raise an issue in the repository.